In [25]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from google.colab import drive

In [26]:
# ==========================================
# 1. MOUNT DRIVE & SETUP PATHS
# ==========================================
print("[SYSTEM] Mounting Google Drive...")
drive.mount('/content/drive')

# Define exact directory structures based on your Drive
base_path = "/content/drive/MyDrive/Colab_Notebooks/LCCAI/"
data_path = os.path.join(base_path, "Datas")
model_path = os.path.join(base_path, "Models")

# Ensure the Models directory exists before we try saving to it
os.makedirs(model_path, exist_ok=True)

[SYSTEM] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
# ==========================================
# 2. LOAD REAL KAGGLE DATA
# ==========================================
print("\n[SYSTEM] Loading ASHRAE building_metadata.csv...")
metadata_file = os.path.join(data_path, "building_metadata.csv")
raw_df = pd.read_csv(metadata_file)
print(f"[SYSTEM] Dataset loaded successfully. Shape: {raw_df.shape}")


[SYSTEM] Loading ASHRAE building_metadata.csv...
[SYSTEM] Dataset loaded successfully. Shape: (1449, 6)


In [28]:
# ==========================================
# 3. SYNTHESIZE FINANCIAL DATA (CPWD PAR-2025)
# ==========================================
print("[SYSTEM] Synthesizing Financial Constraints (India CPWD Baseline)...")

# Clean the data: drop rows missing 'year_built'
df = raw_df.dropna(subset=['year_built']).copy()

# Map Kaggle features to LCCAI Engine schema
df.rename(columns={'square_feet': 'building_area_sqm'}, inplace=True) # Treating sqft as sqm for baseline
df['building_age_years'] = 2026 - df['year_built']

# CPWD Standards (₹45k base cost, 1.5% maintenance)
BASE_COST_PER_SQM = 45000
BASE_MAINTENANCE_RATE = 0.015

# Day 1 Cost
df['initial_construction_cost'] = (df['building_area_sqm'] * BASE_COST_PER_SQM).astype(int)

# Material Quality (Randomized 0.6 - 1.0)
np.random.seed(42)
df['material_quality_index'] = np.random.uniform(0.6, 1.0, len(df))

# 30-Year Targets (What the AI predicts)
df['actual_annual_maintenance'] = (
    (df['initial_construction_cost'] * BASE_MAINTENANCE_RATE) * (1 + (df['building_age_years'] * 0.005)) /
    df['material_quality_index']
).astype(int)

df['actual_annual_energy'] = (df['building_area_sqm'] * 120 * (1 + (df['building_age_years'] * 0.01))).astype(int)
df['climate_zone_severity'] = 3 # Standardized default

[SYSTEM] Synthesizing Financial Constraints (India CPWD Baseline)...


In [29]:
# ==========================================
# 4. TRAIN THE ENGINES
# ==========================================
print("\n[SYSTEM] Initializing Random Forest Training Pipeline...")

X = df[['building_area_sqm', 'building_age_years', 'material_quality_index',
        'initial_construction_cost', 'climate_zone_severity']]
y_maint = df['actual_annual_maintenance']
y_energy = df['actual_annual_energy']

X_train, X_test, y_maint_train, y_maint_test, y_energy_train, y_energy_test = train_test_split(
    X, y_maint, y_energy, test_size=0.2, random_state=42
)

model_maint = RandomForestRegressor(n_estimators=100, random_state=42)
model_maint.fit(X_train, y_maint_train)

model_energy = RandomForestRegressor(n_estimators=100, random_state=42)
model_energy.fit(X_train, y_energy_train)

print(f"  -> Maintenance Accuracy (R²): {r2_score(y_maint_test, model_maint.predict(X_test)):.4f}")
print(f"  -> Energy Accuracy (R²):      {r2_score(y_energy_test, model_energy.predict(X_test)):.4f}")


[SYSTEM] Initializing Random Forest Training Pipeline...
  -> Maintenance Accuracy (R²): 0.9885
  -> Energy Accuracy (R²):      0.9751


In [30]:
# ==========================================
# 5. EXPORT MODELS TO LCCAI/MODELS/
# ==========================================
print(f"\n[SYSTEM] Exporting trained `.pkl` engines to: {model_path}")

maint_export = os.path.join(model_path, "model_maintenance.pkl")
with open(maint_export, "wb") as f:
    pickle.dump(model_maint, f)

energy_export = os.path.join(model_path, "model_energy.pkl")
with open(energy_export, "wb") as f:
    pickle.dump(model_energy, f)

print("[SYSTEM] Pipeline Complete. Models are ready for the Debian local environment.")


[SYSTEM] Exporting trained `.pkl` engines to: /content/drive/MyDrive/Colab_Notebooks/LCCAI/Models
[SYSTEM] Pipeline Complete. Models are ready for the Debian local environment.
